In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

### DATABASE CONNECTION

In [3]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "banking_fraud_dw"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("Connected to database.")

Connected to database.


###  Load New Transactions CSV

In [7]:
file_path = "new_transactions.csv"  # simulate new daily file
df = pd.read_csv(file_path)

print(f"Loaded {len(df)} new records.")

Loaded 5000 new records.


#### Insert into Staging

In [10]:
df.to_sql(
    "transactions",
    engine,
    schema="staging",
    if_exists="append",
    index=False
)
print("Inserted into staging.transactions")

Inserted into staging.transactions


#### Insert into Warehouse Fact Table

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO warehouse.fact_transactions
        SELECT *
        FROM staging.transactions
        WHERE transaction_id NOT IN (
            SELECT transaction_id FROM warehouse.fact_transactions
        );
    """))

print("Inserted into warehouse.fact_transactions")

#### Refresh Aggregation Table

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE warehouse.agg_customer_fraud_summary;"))
    conn.execute(text("""
        INSERT INTO warehouse.agg_customer_fraud_summary
        SELECT 
            customer_id,
            COUNT(*) AS total_transactions,
            SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
            SUM(amount) AS total_amount,
            SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount
        FROM warehouse.fact_transactions
        GROUP BY customer_id;
    """))

print("Aggregation table refreshed.")

#### Refresh Materialized View

In [ ]:
with engine.begin() as conn:
    conn.execute(text("REFRESH MATERIALIZED VIEW warehouse.mv_daily_fraud_stats;"))

print("Materialized view refreshed.")

print("ETL Pipeline Completed Successfully.")